# Kimi K2.5 Thinking vs Instant: World Model Diagnostic

**Goal**: Test whether chain-of-thought reasoning reveals a better internal world model for HNC clinical decisions.

- **32 items** (12 baselines + 15 missed-edge perturbations + 5 nulls + 3 phantom-edge items)
- **5 runs** each at temperature 1.0
- Covers **29/33 missed edges** from the instant model evaluation
- Estimated cost: **~$6**

**Setup**: This notebook mounts your Google Drive and works from the project folder directly. Upload the project folder `Causal Discovery Methods for LLM Evaluation (2)` to your Drive root (or adjust the path in Cell 1).

In [ ]:
# Cell 1: Mount Google Drive & set up working directory
import os, json, time, hashlib, re, random
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── ADJUST THIS PATH if your folder is somewhere else in Drive ──
PROJECT_DIR = Path('/content/drive/MyDrive/Causal Discovery Methods for LLM Evaluation (2)')

# Verify the project folder exists
assert PROJECT_DIR.exists(), f"Project folder not found at {PROJECT_DIR}. Adjust PROJECT_DIR above."

# Change working directory so all relative paths resolve correctly
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

# Verify key files
for f in ['vignette_battery.json', 'response_parser.py', 'kg2_enhanced.py']:
    status = 'OK' if (PROJECT_DIR / f).exists() else 'MISSING'
    print(f'  {f}: {status}')

# Check for instant-mode analysis (needed for comparison in Cells 8-10)
if (PROJECT_DIR / 'analysis_kimi_k25' / 'metrics.json').exists():
    print('  analysis_kimi_k25/: OK')
else:
    print('  analysis_kimi_k25/: MISSING (comparison cells will be skipped)')

# Set your API key
TOGETHER_API_KEY = os.environ.get('TOGETHER_API_KEY', '')
if not TOGETHER_API_KEY:
    TOGETHER_API_KEY = input('Enter Together.ai API key: ')

print(f'\nAPI key set: {TOGETHER_API_KEY[:8]}...')

In [ ]:
# Cell 2: Model config & prompts

MODEL_ID = 'moonshotai/Kimi-K2.5'
MODEL_NAME = 'kimi-k2.5-thinking'
TEMPERATURE = 1.0
MAX_TOKENS = 16384
REASONING_EFFORT = 'high'
TIMEOUT = 600  # 10 min for thinking model

SYSTEM_PROMPT = """You are an expert head and neck surgical oncologist participating in a multidisciplinary tumor board. You have extensive experience with laryngeal and hypopharyngeal cancer management, including transoral laser microsurgery (TLM), open partial horizontal laryngectomies (OPHL), total laryngectomy (TL), concurrent chemoradiotherapy (CRT), and induction chemotherapy (ICT) protocols.

When making treatment recommendations, you should:
- Consider all relevant clinical variables including tumor staging, anatomical extent, functional status, patient fitness, and comorbidities
- Distinguish between absolute contraindications, relative contraindications, and preferences
- Explain your reasoning, specifically which clinical findings drive your recommendations
- When uncertain, express that uncertainty rather than committing to a definitive answer
- Consider both surgical and non-surgical larynx preservation options where appropriate

Respond as you would in a real clinical discussion -- thorough but focused on the specific question asked."""

OPEN_ENDED_TEMPLATE = """Here is a clinical case for multidisciplinary discussion:

{clinical_text}

**Clinical Question**: {question}

Please provide your treatment recommendation with detailed reasoning. For each option you recommend or exclude, explain which specific clinical findings support that decision."""

TARGETED_TEMPLATE = """Based on the same clinical case presented above, please evaluate each treatment option listed below.

For EACH treatment, you MUST use this exact format:

**[Treatment name]**: [APPROPRIATE / CONTRAINDICATED / RELATIVELY CONTRAINDICATED / UNCERTAIN]
Reasoning: [1-2 sentences explaining which clinical findings drive this assessment]

Use ONLY one of these four labels -- do not paraphrase or hedge the label itself. If a treatment is conditionally appropriate (e.g., depends on surgical expertise or exposure), state APPROPRIATE and note the conditions in your reasoning.

{targeted_questions}"""

TARGETED_QUESTIONS = {
    'glottic_cT2': [
        '1. Is transoral laser microsurgery (TLM) appropriate for this patient?',
        '2. Is conventional radiotherapy alone appropriate?',
        '3. Is accelerated or hyperfractionated radiotherapy indicated?',
        '4. Is concurrent chemoradiotherapy warranted?',
    ],
    'glottic_cT3': [
        '1. Is transoral laser microsurgery (TLM) appropriate for this patient?',
        '2. Is open partial horizontal laryngectomy (OPHL type II) appropriate?',
        '3. Is concurrent chemoradiotherapy (CRT) appropriate?',
        '4. Is induction chemotherapy followed by response-adapted treatment appropriate?',
        '5. Is total laryngectomy necessary?',
    ],
    'supraglottic_cT3': [
        '1. Is transoral laser microsurgery (TLM) or transoral robotic surgery (TORS) appropriate?',
        '2. Is supraglottic laryngectomy (OPHL type I) appropriate?',
        '3. Is OPHL type IIB indicated?',
        '4. Is concurrent chemoradiotherapy appropriate?',
        '5. Should partial laryngectomy be avoided given the nodal status?',
    ],
    'hypopharyngeal': [
        '1. Is transoral laser microsurgery (TLM) appropriate for this hypopharyngeal cancer?',
        '2. Is partial laryngectomy appropriate given the tumor extent and nodal disease?',
        '3. Is non-surgical larynx preservation (CRT or ICT+RT) appropriate?',
        '4. Is total laryngectomy the preferred treatment?',
    ],
    'cT4a_unselected': [
        '1. Is total laryngectomy the appropriate treatment?',
        '2. Is any form of larynx preservation viable?',
        '3. Could non-surgical LP (CRT) be considered?',
    ],
    'cT4a_selected': [
        '1. Is total laryngectomy mandatory for this cT4a patient?',
        '2. Is open partial horizontal laryngectomy (OPHL) a viable option?',
        '3. Is non-surgical larynx preservation (CRT) a viable option?',
        '4. Is TLM appropriate?',
    ],
    'cisplatin_eligibility': [
        '1. Is high-dose cisplatin (100 mg/m2 q3w) appropriate for this patient?',
        '2. If not, what is the specific contraindication?',
        '3. Is the contraindication absolute or relative?',
        '4. What alternative concurrent systemic agent would you recommend?',
    ],
    'post_ict_response': [
        '1. Is radiotherapy alone (without concurrent chemotherapy) the appropriate definitive treatment?',
        '2. Is concurrent chemoradiotherapy (CRT) indicated?',
        '3. Should total laryngectomy be recommended?',
        '4. Is continued non-surgical larynx preservation justified based on the ICT response?',
    ],
    'elderly_frail': [
        '1. Does this patient\'s age affect eligibility for TLM?',
        '2. Does this patient\'s age affect eligibility for OPHL?',
        '3. Does this patient\'s age affect eligibility for concurrent CRT?',
        '4. Should treatment be de-escalated or adapted based on age/frailty?',
    ],
    'pretreatment_function': [
        '1. Is non-surgical larynx preservation (CRT or ICT) contraindicated?',
        '2. If contraindicated, is it an absolute or relative contraindication?',
        '3. What specific functional findings drive this assessment?',
        '4. Is total laryngectomy indicated?',
    ],
}

print('Prompts loaded. Model:', MODEL_ID, '(thinking mode, effort:', REASONING_EFFORT, ')')

In [ ]:
# Cell 3: Load battery & select diagnostic subset

with open('vignette_battery.json') as f:
    bat = json.load(f)

# Diagnostic subset: 32 items covering 29/33 missed edges + nulls + phantoms
DIAGNOSTIC_ITEMS = [
    'A1-BASE', 'A1-NULL', 'A1-P1', 'A1-P3',
    'A2-BASE',
    'B1-BASE', 'B1-NULL2', 'B1-P1', 'B1-P2',
    'C1-BASE', 'C1-P3', 'C1-P5',
    'D1-BASE', 'D1-P3',
    'E1-BASE',
    'F1-BASE', 'F1-P1', 'F1-P2', 'F1-P4',
    'G1-ABS11', 'G1-BASE', 'G1-ECOG-TUM', 'G1-GREY-S70', 'G1-NULL',
    'H1-BASE', 'H1-P2', 'H1-P3',
    'H2-BASE',
    'I1-BASE', 'I1-NULL2',
    'J1-BASE', 'J1-NULL',
]

# Build item lookup
items = []
for b in bat['baselines']:
    if b['id'] in DIAGNOSTIC_ITEMS:
        items.append({'id': b['id'], 'family': b['family'], 'type': 'baseline',
                       'clinical_text': b['clinical_text'], 'question': b['question'],
                       'expected_recommendations': b['expected_recommendations'],
                       'expected_excluded': b['expected_excluded']})
for p in bat['perturbations']:
    if p['id'] in DIAGNOSTIC_ITEMS:
        items.append({'id': p['id'], 'family': p['family'], 'type': p['type'],
                       'baseline_id': p['baseline_id'],
                       'clinical_text': p['clinical_text'], 'question': p['question'],
                       'expected_recommendations': p['expected_recommendations'],
                       'expected_excluded': p['expected_excluded'],
                       'edge_justification': p.get('edge_justification', [])})

print(f'Loaded {len(items)} diagnostic items from battery')
families = {}
for item in items:
    f = item['family']
    families[f] = families.get(f, 0) + 1
for f, n in sorted(families.items()):
    print(f'  {f}: {n} items')

In [ ]:
# Cell 4: API call function with robust retry

def call_together_thinking(messages, max_retries=5):
    """Call Together.ai with Kimi K2.5 thinking mode."""
    payload = {
        'model': MODEL_ID,
        'messages': messages,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'reasoning_effort': REASONING_EFFORT,
    }
    headers = {
        'Authorization': f'Bearer {TOGETHER_API_KEY}',
        'Content-Type': 'application/json',
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(
                'https://api.together.xyz/v1/chat/completions',
                headers=headers, json=payload, timeout=TIMEOUT
            )
            if resp.status_code in (503, 429):
                wait = min(2 ** attempt * 15 + random.uniform(0, 10), 180)
                print(f' [503/429, retry in {wait:.0f}s]', end='', flush=True)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            msg = data['choices'][0]['message']
            content = msg.get('content', '') or ''
            reasoning = ''

            # Extract reasoning from reasoning_content field or <think> tags
            if msg.get('reasoning_content'):
                reasoning = msg['reasoning_content'].strip()
            if not reasoning and '<think>' in content:
                m = re.search(r'<think>(.*?)</think>', content, re.DOTALL)
                if m:
                    reasoning = m.group(1).strip()
                    content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()

            return {
                'content': content,
                'reasoning': reasoning,
                'usage': data.get('usage', {}),
                'finish_reason': data['choices'][0].get('finish_reason', 'unknown'),
                'error': None,
            }
        except requests.exceptions.ReadTimeout:
            if attempt < max_retries - 1:
                wait = min(2 ** attempt * 20 + random.uniform(0, 15), 240)
                print(f' [timeout, retry in {wait:.0f}s]', end='', flush=True)
                time.sleep(wait)
                continue
            return {'content': '', 'reasoning': '', 'usage': {},
                    'finish_reason': 'error', 'error': 'ReadTimeout after all retries'}
        except Exception as e:
            return {'content': '', 'reasoning': '', 'usage': {},
                    'finish_reason': 'error', 'error': f'{type(e).__name__}: {str(e)[:200]}'}

    return {'content': '', 'reasoning': '', 'usage': {},
            'finish_reason': 'error', 'error': 'Max retries exceeded'}

print('API function ready.')

In [ ]:
# Cell 5: Run the diagnostic battery

N_RUNS = 5
N_WORKERS = 3  # Conservative for thinking model (longer responses)
NL = chr(10)
OUTDIR = Path('results_kimi_thinking')
OUTDIR.mkdir(exist_ok=True)

# Checkpoint file
cp = OUTDIR / f'run_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")}.jsonl'

# Scan existing checkpoints for completed queries
done_ok = set()
done_err = set()
for jf in OUTDIR.glob('run_*.jsonl'):
    with open(jf) as f:
        for line in f:
            if line.strip():
                try:
                    r = json.loads(line)
                    h = r.get('hash', '')
                    if r.get('error') is None:
                        done_ok.add(h)
                    else:
                        done_err.add(h)
                except:
                    pass

def make_hash(item_id, run):
    return hashlib.md5(f'{item_id}:{MODEL_NAME}:{run}'.encode()).hexdigest()[:12]

# Build work queue (skip completed)
work = []
for item in items:
    for run_idx in range(N_RUNS):
        h = make_hash(item['id'], run_idx)
        if h not in done_ok:
            work.append((item, run_idx, h))

print(f'Total: {len(items)} items x {N_RUNS} runs = {len(items) * N_RUNS} queries')
print(f'Already done: {len(done_ok)} ok, {len(done_err)} errors')
print(f'Remaining: {len(work)}')
print(f'Checkpoint: {cp}')

if not work:
    print('All done!')
else:
    def process_one(item, run_idx, h):
        """Process a single item: Phase 1 + Phase 2."""
        result = {
            'item_id': item['id'], 'model_name': MODEL_NAME,
            'model_id': MODEL_ID, 'run_idx': run_idx, 'hash': h,
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'temperature': TEMPERATURE, 'family': item['family'],
            'item_type': item.get('type', 'baseline'), 'tier': 'reasoning_small',
            'phase1_response': None, 'phase1_reasoning': None,
            'phase2_response': None, 'phase2_reasoning': None,
            'phase1_usage': None, 'phase2_usage': None, 'error': None,
        }
        try:
            # Phase 1: Open-ended
            msgs1 = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': OPEN_ENDED_TEMPLATE.format(
                    clinical_text=item['clinical_text'].strip(),
                    question=item['question'].strip())},
            ]
            r1 = call_together_thinking(msgs1)
            if r1['error']:
                result['error'] = f'Phase1: {r1["error"]}'
                return result
            result['phase1_response'] = r1['content']
            result['phase1_reasoning'] = r1['reasoning']
            result['phase1_usage'] = r1['usage']

            # Phase 2: Structured
            qs = TARGETED_QUESTIONS.get(item['family'], [
                '1. Which treatments are appropriate?',
                '2. Which treatments are contraindicated?',
            ])
            msgs2 = msgs1 + [
                {'role': 'assistant', 'content': r1['content']},
                {'role': 'user', 'content': TARGETED_TEMPLATE.format(
                    targeted_questions=NL.join(qs))},
            ]
            r2 = call_together_thinking(msgs2)
            if r2['error']:
                result['error'] = f'Phase2: {r2["error"]}'
                return result
            result['phase2_response'] = r2['content']
            result['phase2_reasoning'] = r2['reasoning']
            result['phase2_usage'] = r2['usage']

        except Exception as e:
            result['error'] = f'{type(e).__name__}: {str(e)[:200]}'
        return result

    # Run with thread pool
    t0 = time.time()
    n_done = 0
    n_err = 0
    fout = open(cp, 'a')

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {}
        for i, (item, run_idx, h) in enumerate(work):
            # Stagger submissions
            if i > 0 and i % N_WORKERS == 0:
                time.sleep(2)
            fut = pool.submit(process_one, item, run_idx, h)
            futures[fut] = (item['id'], run_idx)

        for fut in as_completed(futures):
            item_id, run_idx = futures[fut]
            n_done += 1
            res = fut.result()
            fout.write(json.dumps(res, default=str) + NL)
            fout.flush()

            elapsed = time.time() - t0
            rate = n_done / elapsed * 60 if elapsed > 0 else 0
            eta = (len(work) - n_done) / rate if rate > 0 else 0

            if res['error']:
                n_err += 1
                print(f'[{n_done}/{len(work)}] {item_id} r{run_idx} X {res["error"][:60]} | {rate:.1f}/min ETA {eta:.0f}m')
            else:
                p1_len = len(res['phase1_response'] or '')
                p2_len = len(res['phase2_response'] or '')
                think_len = len(res['phase1_reasoning'] or '')
                print(f'[{n_done}/{len(work)}] {item_id} r{run_idx} ok ({p1_len}+{p2_len}c, think:{think_len}c) | {rate:.1f}/min ETA {eta:.0f}m')

    fout.close()
    print(f'{NL}Done: {n_done} in {(time.time()-t0)/60:.1f}m, {n_err} errors')
    print(f'Results: {cp}')

In [ ]:
# Cell 6: Quick sanity check - inspect a few results

results = []
for jf in sorted(OUTDIR.glob('run_*.jsonl')):
    with open(jf) as f:
        for line in f:
            if line.strip():
                try:
                    results.append(json.loads(line))
                except:
                    pass

print(f'Total results: {len(results)}')
print(f'Errors: {sum(1 for r in results if r.get("error"))}')

# Show a sample with thinking chain
for r in results[:3]:
    if r.get('error') is None:
        print(f'{NL}{"="*60}')
        print(f'{r["item_id"]} run {r["run_idx"]}')
        think = r.get('phase1_reasoning', '')
        if think:
            print(f'THINKING ({len(think)} chars): {think[:500]}...')
        p1 = r.get('phase1_response', '')
        print(f'RESPONSE ({len(p1)} chars): {p1[:500]}...')
        break

In [ ]:
# Cell 7: Run analysis - parse stances & compare with instant model
# This uses the same response_parser.py from the main pipeline

import sys
sys.path.insert(0, '.')
from response_parser import extract_stances, run_analysis

# Merge all result files into one for analysis
merged = OUTDIR / 'merged_results.jsonl'
seen = set()
with open(merged, 'w') as out:
    for jf in sorted(OUTDIR.glob('run_*.jsonl')):
        with open(jf) as f:
            for line in f:
                if line.strip():
                    try:
                        r = json.loads(line)
                        h = r.get('hash', '')
                        if h not in seen and r.get('error') is None:
                            seen.add(h)
                            out.write(line)
                    except:
                        pass

print(f'Merged {len(seen)} unique successful results')

# Run standard analysis
analysis_dir = Path('analysis_kimi_k25_thinking')
analysis_dir.mkdir(exist_ok=True)

run_analysis(
    results_path=str(merged),
    battery_path='vignette_battery.json',
    output_dir=str(analysis_dir)
)
print(f'Analysis saved to {analysis_dir}/')

In [ ]:
# Cell 8: Head-to-head comparison: Thinking vs Instant

# Load both sets of metrics
def load_metrics(path):
    with open(path) as f:
        return json.load(f)

thinking_metrics = load_metrics(analysis_dir / 'metrics.json')
instant_metrics = load_metrics('analysis_kimi_k25/metrics.json')

print('=' * 70)
print('KIMI K2.5: THINKING vs INSTANT MODE COMPARISON')
print('=' * 70)
print(f'{"Metric":<35s} {"Instant":>12s} {"Thinking":>12s} {"Delta":>10s}')
print('-' * 70)

compare_keys = [
    ('rec_accuracy', 'Recommendation Accuracy'),
    ('exc_accuracy', 'Exclusion Accuracy'),
    ('rec_precision', 'Recommendation Precision'),
    ('exc_precision', 'Exclusion Precision'),
    ('rec_recall', 'Recommendation Recall'),
    ('exc_recall', 'Exclusion Recall'),
]

for key, label in compare_keys:
    iv = instant_metrics.get(key, 0)
    tv = thinking_metrics.get(key, 0)
    delta = tv - iv
    arrow = '+' if delta > 0 else ''
    print(f'{label:<35s} {iv:>11.1%} {tv:>11.1%} {arrow}{delta:>9.1%}')

In [ ]:
# Cell 9: Edge-level comparison (the key question: does thinking recover more edges?)

# Load KG2 analysis for both
try:
    from kg2_enhanced import run_kg2_analysis
    run_kg2_analysis(
        parsed_path=str(analysis_dir / 'parsed.json'),
        battery_path='vignette_battery.json',
        output_dir=str(analysis_dir)
    )
    print('KG2 analysis complete')
except Exception as e:
    print(f'KG2 analysis error (may need full battery): {e}')
    print('Falling back to basic comparison...')

# Load graph comparison if available
for model_label, adir in [('Instant', 'analysis_kimi_k25'), ('Thinking', str(analysis_dir))]:
    gc_path = Path(adir) / 'graph_comparison.json'
    if gc_path.exists():
        with open(gc_path) as f:
            gc = json.load(f)
        model_key = list(gc.keys())[0]
        d = gc[model_key]
        print(f'{NL}{model_label} ({model_key}):')
        print(f'  Hard TP: {d["true_positives"]}, Soft TP: {d["soft_true_positives"]}')
        print(f'  FN: {d["false_negatives"]}, FP: {d["false_positives"]}')
        print(f'  Soft F1: {d["soft_f1"]:.3f}')
        print(f'  Null JSD 95th: {d["null_jsd_95"]:.4f}')
    else:
        print(f'{NL}{model_label}: graph_comparison.json not found')

In [ ]:
# Cell 10: Per-item stance comparison (thinking vs instant)
# Focus on the diagnostic items to see where thinking helps

# Load parsed results
thinking_parsed_path = analysis_dir / 'parsed.json'
instant_parsed_path = Path('analysis_kimi_k25/parsed.json')

if thinking_parsed_path.exists() and instant_parsed_path.exists():
    with open(thinking_parsed_path) as f:
        tp = json.load(f)
    with open(instant_parsed_path) as f:
        ip = json.load(f)

    STANCE_MAP = {'recommended': 'APP', 'excluded': 'CI',
                  'relative_ci': 'RCI', 'uncertain': 'UNC'}

    def get_stance_dist(parsed, item_id):
        recs = [r for r in parsed if r['item_id'] == item_id]
        dist = {}
        for r in recs:
            for s in r.get('stances', []):
                t = s['treatment']
                st = STANCE_MAP.get(s['stance'], s['stance'])
                if t not in dist:
                    dist[t] = {}
                dist[t][st] = dist[t].get(st, 0) + 1
        return dist, len(recs)

    # Compare key diagnostic items
    print('PER-ITEM THINKING vs INSTANT COMPARISON')
    print('=' * 80)

    for item_id in DIAGNOSTIC_ITEMS:
        i_dist, i_n = get_stance_dist(ip, item_id)
        t_dist, t_n = get_stance_dist(tp, item_id)

        if t_n == 0:
            continue  # No thinking results for this item

        all_treats = sorted(set(list(i_dist.keys()) + list(t_dist.keys())))
        has_diff = False
        for treat in all_treats:
            i_total = sum(i_dist.get(treat, {}).values())
            t_total = sum(t_dist.get(treat, {}).values())
            if i_total == 0 and t_total == 0:
                continue
            i_app = i_dist.get(treat, {}).get('APP', 0) / i_total * 100 if i_total else 0
            t_app = t_dist.get(treat, {}).get('APP', 0) / t_total * 100 if t_total else 0
            if abs(i_app - t_app) > 20:
                has_diff = True
                break

        if has_diff:
            print(f'{NL}{item_id} (instant n={i_n}, thinking n={t_n})')
            for treat in all_treats:
                i_total = sum(i_dist.get(treat, {}).values())
                t_total = sum(t_dist.get(treat, {}).values())
                if i_total == 0 and t_total == 0:
                    continue
                i_app = i_dist.get(treat, {}).get('APP', 0) / i_total * 100 if i_total else 0
                t_app = t_dist.get(treat, {}).get('APP', 0) / t_total * 100 if t_total else 0
                i_ci = i_dist.get(treat, {}).get('CI', 0) / i_total * 100 if i_total else 0
                t_ci = t_dist.get(treat, {}).get('CI', 0) / t_total * 100 if t_total else 0
                shift = abs(i_app - t_app)
                marker = ' <<<' if shift > 30 else ''
                print(f'  {treat:22s} Inst: APP={i_app:5.1f}% CI={i_ci:5.1f}%  |  Think: APP={t_app:5.1f}% CI={t_ci:5.1f}%{marker}')

    print(f'{NL}(Only showing items with >20% shift in at least one treatment)')
else:
    print('Parsed results not found. Run Cell 7 first.')

In [ ]:
# Cell 11: Thinking chain quality analysis
# Examine what the model actually reasons about

results = []
for jf in sorted(OUTDIR.glob('run_*.jsonl')):
    with open(jf) as f:
        for line in f:
            if line.strip():
                try:
                    r = json.loads(line)
                    if r.get('error') is None:
                        results.append(r)
                except:
                    pass

# Analyze thinking chain characteristics
think_lengths = [len(r.get('phase1_reasoning', '') or '') for r in results]
resp_lengths = [len(r.get('phase1_response', '') or '') for r in results]

print('THINKING CHAIN STATISTICS')
print(f'  Mean thinking length: {sum(think_lengths)/len(think_lengths):.0f} chars')
print(f'  Mean response length: {sum(resp_lengths)/len(resp_lengths):.0f} chars')
print(f'  Think/Response ratio: {sum(think_lengths)/max(sum(resp_lengths),1):.1f}x')

# Check if thinking mentions key clinical concepts
key_concepts = ['cartilage', 'exposure', 'cisplatin', 'CrCl', 'creatinine',
                'laryngectomy', 'preservation', 'contraindicated', 'T stage',
                'OPHL', 'TLM', 'arytenoid', 'paraglottic']

print(f'{NL}KEY CONCEPT FREQUENCY IN THINKING CHAINS:')
for concept in key_concepts:
    count = sum(1 for r in results
                if concept.lower() in (r.get('phase1_reasoning', '') or '').lower())
    pct = count / len(results) * 100 if results else 0
    print(f'  {concept:20s}: {count:3d}/{len(results)} ({pct:.0f}%)')

# Show best thinking chain example
longest = max(results, key=lambda r: len(r.get('phase1_reasoning', '') or ''))
print(f'{NL}{"="*60}')
print(f'LONGEST THINKING CHAIN: {longest["item_id"]} run {longest["run_idx"]}')
think = longest.get('phase1_reasoning', '')
print(f'Length: {len(think)} chars')
print(f'First 1000 chars:{NL}{think[:1000]}')

In [ ]:
# Cell 12: Summary verdict

print('=' * 70)
print('VERDICT: Does thinking mode reveal a better world model?')
print('=' * 70)
print()
print('Key questions to answer from the data above:')
print()
print('1. EDGE RECOVERY: Does thinking detect more causal edges?')
print('   -> Compare soft TP counts (Cell 9)')
print()
print('2. ACCURACY: Are treatment stances more aligned with expert consensus?')
print('   -> Compare rec/exc accuracy (Cell 8)')
print()
print('3. STABILITY: Are null perturbations more stable?')
print('   -> Compare null JSD 95th percentile (Cell 9)')
print()
print('4. REASONING QUALITY: Does the thinking chain mention the right variables?')
print('   -> Check concept frequency (Cell 11)')
print()
print('If thinking improves all 4: knowledge is latent, reasoning unlocks it.')
print('If thinking improves 1-2: partial world model, gaps remain.')
print('If no improvement: the world model is truly absent from pretraining.')